# Задание 5. Марковские цепи: матрица переходов для ДНК 1-го порядка

In [1]:
from Bio import SeqIO
import numpy as np
import pandas as pd

In [7]:
# Загрузка последовательности
record = SeqIO.read("/Users/veronikaaksinina/Documents/bioinf_sem_26/chr21.fasta", "fasta")
seq = str(record.seq)

## 1. Подсчёт частот динуклеотидов

In [8]:
bases = ['A', 'C', 'G', 'T']

# Считаем все 16 динуклеотидов
dinuc_counts = {b1 + b2: 0 for b1 in bases for b2 in bases}

for i in range(len(seq) - 1):
    dinuc = seq[i] + seq[i+1]
    if dinuc in dinuc_counts:
        dinuc_counts[dinuc] += 1

print("Частоты динуклеотидов:")
for dinuc, count in dinuc_counts.items():
    print(f"  {dinuc}: {count}")

Частоты динуклеотидов:
  AA: 3915117
  AC: 2038442
  AG: 2774777
  AT: 3092310
  CA: 2921760
  CC: 2043408
  CG: 462299
  CT: 2757749
  GA: 2420557
  GC: 1705746
  GG: 2062676
  GT: 2037399
  TA: 2563211
  TC: 2397643
  TG: 2926603
  TT: 3968871


## 2. Матрица переходов 4×4

In [9]:
# Строим матрицу: P[i][j] = P(j | i)
transition_matrix = np.zeros((4, 4))

for i, b1 in enumerate(bases):
    row_total = sum(dinuc_counts[b1 + b2] for b2 in bases)
    for j, b2 in enumerate(bases):
        if row_total > 0:
            transition_matrix[i, j] = dinuc_counts[b1 + b2] / row_total

df = pd.DataFrame(transition_matrix, index=bases, columns=bases)
print("Матрица переходов P:")
print(df.round(4))

Матрица переходов P:
        A       C       G       T
A  0.3312  0.1724  0.2347  0.2616
C  0.3570  0.2496  0.0565  0.3369
G  0.2942  0.2074  0.2507  0.2477
T  0.2162  0.2022  0.2468  0.3347


## 3. Проверка: сумма строк равна 1

In [10]:
row_sums = transition_matrix.sum(axis=1)
print("Суммы строк матрицы переходов:")
for b, s in zip(bases, row_sums):
    print(f"  {b}: {s:.6f}")

Суммы строк матрицы переходов:
  A: 1.000000
  C: 1.000000
  G: 1.000000
  T: 1.000000


## 4. Стационарное распределение (π = πP)

In [11]:
# Решаем через собственные векторы: π — левый собственный вектор для λ=1
# Это эквивалентно правому собственному вектору транспонированной матрицы
eigenvalues, eigenvectors = np.linalg.eig(transition_matrix.T)

# Находим собственный вектор для λ ≈ 1
idx = np.argmin(np.abs(eigenvalues - 1.0))
stationary = np.real(eigenvectors[:, idx])
stationary = stationary / stationary.sum()  # нормируем

print("Стационарное распределение π:")
for b, p in zip(bases, stationary):
    print(f"  π({b}) = {p:.4f}")

Стационарное распределение π:
  π(A) = 0.2949
  π(C) = 0.2042
  π(G) = 0.2052
  π(T) = 0.2958


## 5. Сравнение со наблюдаемыми частотами

In [12]:
# Наблюдаемые частоты нуклеотидов
observed = {b: seq.count(b) / len(seq) for b in bases}

print(f"{'Нуклеотид':<12} {'Наблюдаемая':>15} {'Стационарная':>15} {'Разница':>10}")
print("-" * 55)
for b, pi in zip(bases, stationary):
    obs = observed[b]
    diff = abs(obs - pi)
    print(f"{b:<12} {obs:>15.4f} {pi:>15.4f} {diff:>10.4f}")

Нуклеотид        Наблюдаемая    Стационарная    Разница
-------------------------------------------------------
A                     0.2531          0.2949     0.0418
C                     0.1752          0.2042     0.0289
G                     0.1761          0.2052     0.0291
T                     0.2538          0.2958     0.0419
